# Chapter 6: Transforms with Polars

**Lazy execution • Chaining without explosions • DuckDB <-> Polars handoff**

This notebook demonstrates:
- Lazy execution vs eager evaluation
- Method chaining patterns
- Zero-copy DuckDB ↔ Polars integration
- Financial reconciliation with validation
- Streaming processing for large datasets

## Setup

In [1]:
import polars as pl
import duckdb
from datetime import datetime, timedelta
from typing import List, Tuple
from pathlib import Path

# Create data directories
Path("data/curated/reconciliation").mkdir(parents=True, exist_ok=True)
Path("data/curated/validation").mkdir(parents=True, exist_ok=True)

print(f"Polars version: {pl.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

Polars version: 1.34.0
DuckDB version: 1.4.0


## 1. Pandas vs Polars Comparison

### Key Differences:
1. **Lazy execution**: Builds query plan, optimizes, then runs once
2. **No copies**: Arrow-native; zero-copy to/from DuckDB
3. **Parallel by default**: Uses all cores without `.apply()` hell

In [2]:
# Create sample transaction data
sample_data = pl.DataFrame({
    "transaction_id": [f"TXN{i:04d}" for i in range(1000)],
    "amount": [float(i % 100 + 10) for i in range(1000)],
})

# Save as Parquet for example
sample_data.write_parquet("data/raw/transactions.parquet")

# Polars lazy approach
df = (
    pl.scan_parquet("data/raw/transactions.parquet")
    .filter(pl.col("amount") > 0)
    .with_columns((pl.col("amount") * 0.029).alias("fee"))
    .unique(subset=["transaction_id"])
)

print("Lazy dataframe created (not executed yet)")
print(f"Type: {type(df)}")

# Execute and show results
result = df.collect()
print(f"\nExecuted! Shape: {result.shape}")
print(result.head())

Lazy dataframe created (not executed yet)
Type: <class 'polars.lazyframe.frame.LazyFrame'>

Executed! Shape: (1000, 3)
shape: (5, 3)
┌────────────────┬────────┬───────┐
│ transaction_id ┆ amount ┆ fee   │
│ ---            ┆ ---    ┆ ---   │
│ str            ┆ f64    ┆ f64   │
╞════════════════╪════════╪═══════╡
│ TXN0844        ┆ 54.0   ┆ 1.566 │
│ TXN0356        ┆ 66.0   ┆ 1.914 │
│ TXN0433        ┆ 43.0   ┆ 1.247 │
│ TXN0355        ┆ 65.0   ┆ 1.885 │
│ TXN0872        ┆ 82.0   ┆ 2.378 │
└────────────────┴────────┴───────┘


## 2. Lazy Execution Mental Model

**Lazy execution** defers computation until results are requested. Polars builds an execution plan that can be optimized before running.

In [3]:
# Create sample sales data
sales_data = pl.DataFrame({
    "region": ["US", "EU", "US", "APAC", "EU", "US"] * 1000,
    "product_id": [f"P{i % 10:03d}" for i in range(6000)],
    "revenue": [float((i * 7 + 13) % 500 + 100) for i in range(6000)],
})

sales_data.write_parquet("data/raw/sales.parquet")

# This doesn't run yet
lazy_df = (
    pl.scan_parquet("data/raw/sales.parquet")
    .filter(pl.col("region") == "US")
    .group_by("product_id")
    .agg(pl.col("revenue").sum())
    .sort("revenue", descending=True)
)

print("Lazy plan created:")
print("1. Scan only 'region' column (projection pushdown)")
print("2. Filter at file level (predicate pushdown)")
print("3. Group, aggregate, sort in one pass\n")

# Now execute
result = lazy_df.collect()
print(f"Executed! Top 5 products by revenue:")
print(result.head())

# Fast sampling without full execution
print("\nFast sampling (fetch 5 rows):")
sample = lazy_df.fetch(5)
print(sample)

Lazy plan created:
1. Scan only 'region' column (projection pushdown)
2. Filter at file level (predicate pushdown)
3. Group, aggregate, sort in one pass

Executed! Top 5 products by revenue:
shape: (5, 2)
┌────────────┬──────────┐
│ product_id ┆ revenue  │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ P008       ┆ 141600.0 │
│ P002       ┆ 140800.0 │
│ P006       ┆ 140000.0 │
│ P000       ┆ 139200.0 │
│ P004       ┆ 138400.0 │
└────────────┴──────────┘

Fast sampling (fetch 5 rows):
shape: (5, 2)
┌────────────┬──────────┐
│ product_id ┆ revenue  │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ P008       ┆ 141600.0 │
│ P002       ┆ 140800.0 │
│ P006       ┆ 140000.0 │
│ P000       ┆ 139200.0 │
│ P004       ┆ 138400.0 │
└────────────┴──────────┘


/var/folders/cd/l_bbk7dd2fbdr7b_mtcr971m0000gn/T/ipykernel_74379/255572409.py:31: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  sample = lazy_df.fetch(5)


## 3. Chaining Patterns

### Bad: Intermediate Variables (Lost Optimization)

In [4]:
# Create sample orders data
orders = pl.DataFrame({
    "order_id": range(1000),
    "customer_id": [(i % 100) + 1 for i in range(1000)],
    "status": ["paid" if i % 3 == 0 else "pending" for i in range(1000)],
    "total": [float(i % 200 + 50) for i in range(1000)],
})

customers = pl.DataFrame({
    "customer_id": range(1, 101),
    "customer_name": [f"Customer {i}" for i in range(1, 101)],
})

orders.write_parquet("data/raw/orders.parquet")

# BAD: Breaking the chain
df1 = pl.scan_parquet("data/raw/orders.parquet")
df2 = df1.filter(pl.col("status") == "paid")
df3 = df2.with_columns(pl.col("total").cast(pl.Float64))
df4 = df3.join(customers.lazy(), on="customer_id")
result_bad = df4.collect()
print("BAD: Polars can't optimize across these breaks")
print(f"Result shape: {result_bad.shape}\n")

BAD: Polars can't optimize across these breaks
Result shape: (334, 5)



### Good: Single Chain (Full Optimization)

In [5]:
# GOOD: Single chain allows full optimization
result_good = (
    pl.scan_parquet("data/raw/orders.parquet")
    .filter(pl.col("status") == "paid")
    .with_columns(pl.col("total").cast(pl.Float64))
    .join(customers.lazy(), on="customer_id", how="left")
    .collect()
)

print("GOOD: Single chain enables full query optimization")
print(f"Result shape: {result_good.shape}")
print(result_good.head())

GOOD: Single chain enables full query optimization
Result shape: (334, 5)
shape: (5, 5)
┌──────────┬─────────────┬────────┬───────┬───────────────┐
│ order_id ┆ customer_id ┆ status ┆ total ┆ customer_name │
│ ---      ┆ ---         ┆ ---    ┆ ---   ┆ ---           │
│ i64      ┆ i64         ┆ str    ┆ f64   ┆ str           │
╞══════════╪═════════════╪════════╪═══════╪═══════════════╡
│ 0        ┆ 1           ┆ paid   ┆ 50.0  ┆ Customer 1    │
│ 3        ┆ 4           ┆ paid   ┆ 53.0  ┆ Customer 4    │
│ 6        ┆ 7           ┆ paid   ┆ 56.0  ┆ Customer 7    │
│ 9        ┆ 10          ┆ paid   ┆ 59.0  ┆ Customer 10   │
│ 12       ┆ 13          ┆ paid   ┆ 62.0  ┆ Customer 13   │
└──────────┴─────────────┴────────┴───────┴───────────────┘


### When to Break the Chain (Debugging)

In [6]:
# Step 1: Get filtered base
base = (
    pl.scan_parquet("data/raw/orders.parquet")
    .filter(pl.col("status") == "paid")
    .collect()
)
print(f"Debug checkpoint - Filtered rows: {len(base)}")

# Step 2: Continue chain
result = (
    base.lazy()  # Back to lazy mode
    .with_columns(pl.col("total").cast(pl.Float64))
    .join(customers.lazy(), on="customer_id", how="left")
    .collect()
)

print(f"Final result shape: {result.shape}")

Debug checkpoint - Filtered rows: 334
Final result shape: (334, 5)


## 4. DuckDB <-> Polars Handoff (Zero-Copy)

**Zero-copy data transfer** means data is shared between DuckDB and Polars without duplicating it in memory.

### Pattern 1: DuckDB for complex SQL, Polars for transforms

In [7]:
# Create sample events data
events = pl.DataFrame({
    "user_id": [1, 1, 1, 2, 2, 3, 3, 3, 3],
    "session_id": ["S1", "S1", "S2", "S3", "S3", "S4", "S4", "S4", "S5"],
    "event_time": [
        datetime(2024, 10, 1, 10, 0, 0),
        datetime(2024, 10, 1, 10, 5, 0),
        datetime(2024, 10, 1, 11, 0, 0),
        datetime(2024, 10, 1, 9, 30, 0),
        datetime(2024, 10, 1, 9, 45, 0),
        datetime(2024, 10, 1, 14, 0, 0),
        datetime(2024, 10, 1, 14, 20, 0),
        datetime(2024, 10, 1, 14, 25, 0),
        datetime(2024, 10, 1, 15, 30, 0),
    ],
    "event_date": [datetime(2024, 10, 1).date()] * 9,
})

# DuckDB: Complex window functions
con = duckdb.connect()
arrow_table = con.execute("""
    SELECT
        user_id,
        session_id,
        event_time,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) as prev_event_time
    FROM events
    WHERE event_date = '2024-10-01'
""").fetch_arrow_table()

# Polars: Fast column operations
df = pl.from_arrow(arrow_table)
df = df.with_columns(
    (pl.col("event_time") - pl.col("prev_event_time"))
    .dt.total_seconds()
    .alias("session_gap_seconds")
).filter(pl.col("session_gap_seconds") > 1800)

print("Session gaps > 30 minutes:")
print(df)

# Back to DuckDB for final aggregation
con.execute("""
    CREATE TABLE session_stats AS
    SELECT * FROM df
""")

stats = con.execute("SELECT user_id, COUNT(*) as gap_count FROM session_stats GROUP BY user_id").df()
print("\nUser session gap statistics:")
print(stats)

Session gaps > 30 minutes:
shape: (2, 5)
┌─────────┬────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│ user_id ┆ session_id ┆ event_time          ┆ prev_event_time     ┆ session_gap_seconds │
│ ---     ┆ ---        ┆ ---                 ┆ ---                 ┆ ---                 │
│ i64     ┆ str        ┆ datetime[μs]        ┆ datetime[μs]        ┆ i64                 │
╞═════════╪════════════╪═════════════════════╪═════════════════════╪═════════════════════╡
│ 3       ┆ S5         ┆ 2024-10-01 15:30:00 ┆ 2024-10-01 14:25:00 ┆ 3900                │
│ 1       ┆ S2         ┆ 2024-10-01 11:00:00 ┆ 2024-10-01 10:05:00 ┆ 3300                │
└─────────┴────────────┴─────────────────────┴─────────────────────┴─────────────────────┘

User session gap statistics:
   user_id  gap_count
0        1          1
1        3          1


### Pattern 2: Polars for ETL, DuckDB for serving

In [8]:
# Create sample customer data
customer_data = pl.DataFrame({
    "email": ["Alice@Example.com", "bob@test.COM", "Charlie@DEMO.org"],
    "revenue": [1500.0, None, 2300.0],
    "country": ["US", None, "CA"],
})

customer_data.write_csv("data/raw/customers.csv")

# Transform in Polars
cleaned = (
    pl.scan_csv("data/raw/customers.csv")
    .with_columns([
        pl.col("email").str.to_lowercase().alias("email_clean"),
        pl.col("revenue").fill_null(0),
        pl.when(pl.col("country").is_null())
          .then(pl.lit("UNKNOWN"))
          .otherwise(pl.col("country"))
          .alias("country_clean")
    ])
    .collect()
)

print("Cleaned data:")
print(cleaned)

# Write to Parquet
cleaned.write_parquet("data/curated/customers.parquet")

# Query with DuckDB
result = duckdb.sql("""
    SELECT country_clean, COUNT(*) as customer_count, SUM(revenue) as total_revenue
    FROM 'data/curated/customers.parquet'
    GROUP BY country_clean
""")

print("\nAggregated by country:")
print(result.df())

Cleaned data:
shape: (3, 5)
┌───────────────────┬─────────┬─────────┬───────────────────┬───────────────┐
│ email             ┆ revenue ┆ country ┆ email_clean       ┆ country_clean │
│ ---               ┆ ---     ┆ ---     ┆ ---               ┆ ---           │
│ str               ┆ f64     ┆ str     ┆ str               ┆ str           │
╞═══════════════════╪═════════╪═════════╪═══════════════════╪═══════════════╡
│ Alice@Example.com ┆ 1500.0  ┆ US      ┆ alice@example.com ┆ US            │
│ bob@test.COM      ┆ 0.0     ┆ null    ┆ bob@test.com      ┆ UNKNOWN       │
│ Charlie@DEMO.org  ┆ 2300.0  ┆ CA      ┆ charlie@demo.org  ┆ CA            │
└───────────────────┴─────────┴─────────┴───────────────────┴───────────────┘

Aggregated by country:
  country_clean  customer_count  total_revenue
0            US               1         1500.0
1       UNKNOWN               1            0.0
2            CA               1         2300.0


## 5. Financial Reconciliation with Validation

**Financial reconciliation** matches transactions between two systems to ensure consistency.

### Setup: Sample Data

In [9]:
# Bank transactions (external)
bank = pl.DataFrame({
    "bank_id": ["B001", "B002", "B003", "B004"],
    "amount": [1500.00, 2300.50, 890.25, 1500.00],
    "timestamp": [
        datetime(2024, 10, 1, 9, 15, 30),
        datetime(2024, 10, 1, 14, 22, 10),
        datetime(2024, 10, 2, 11, 5, 45),
        datetime(2024, 10, 2, 16, 30, 20),
    ],
    "description": ["Invoice #1234", "Payment XYZ", "Refund", "Invoice #1234"]
})

# Internal ledger (our records)
ledger = pl.DataFrame({
    "ledger_id": ["L001", "L002", "L003"],
    "amount": [1500.00, 2300.50, 995.00],
    "timestamp": [
        datetime(2024, 10, 1, 9, 16, 0),   # 30s diff
        datetime(2024, 10, 1, 14, 22, 15), # 5s diff
        datetime(2024, 10, 2, 10, 58, 30), # No match
    ],
    "reference": ["INV-1234", "PAY-XYZ", "SAL-995"]
})

print("Bank transactions:")
print(bank)
print("\nLedger entries:")
print(ledger)

Bank transactions:
shape: (4, 4)
┌─────────┬────────┬─────────────────────┬───────────────┐
│ bank_id ┆ amount ┆ timestamp           ┆ description   │
│ ---     ┆ ---    ┆ ---                 ┆ ---           │
│ str     ┆ f64    ┆ datetime[μs]        ┆ str           │
╞═════════╪════════╪═════════════════════╪═══════════════╡
│ B001    ┆ 1500.0 ┆ 2024-10-01 09:15:30 ┆ Invoice #1234 │
│ B002    ┆ 2300.5 ┆ 2024-10-01 14:22:10 ┆ Payment XYZ   │
│ B003    ┆ 890.25 ┆ 2024-10-02 11:05:45 ┆ Refund        │
│ B004    ┆ 1500.0 ┆ 2024-10-02 16:30:20 ┆ Invoice #1234 │
└─────────┴────────┴─────────────────────┴───────────────┘

Ledger entries:
shape: (3, 4)
┌───────────┬────────┬─────────────────────┬───────────┐
│ ledger_id ┆ amount ┆ timestamp           ┆ reference │
│ ---       ┆ ---    ┆ ---                 ┆ ---       │
│ str       ┆ f64    ┆ datetime[μs]        ┆ str       │
╞═══════════╪════════╪═════════════════════╪═══════════╡
│ L001      ┆ 1500.0 ┆ 2024-10-01 09:16:00 ┆ INV-1234  │
│ L0

### Step 1: Normalize and Add Match Keys

In [10]:
def prepare_for_matching(df, id_col):
    return df.with_columns([
        # Round amount to cents (avoid float precision issues)
        (pl.col("amount") * 100).round(0).cast(pl.Int64).alias("amount_cents"),
        # Extract date for exact day matching
        pl.col("timestamp").dt.date().alias("date"),
        # Time as seconds since midnight (for fuzzy match)
        (
            pl.col("timestamp").dt.hour() * 3600 +
            pl.col("timestamp").dt.minute() * 60 +
            pl.col("timestamp").dt.second()
        ).alias("time_seconds")
    ])

bank_prep = prepare_for_matching(bank, "bank_id")
ledger_prep = prepare_for_matching(ledger, "ledger_id")

print("Bank transactions (prepared):")
print(bank_prep.select(["bank_id", "amount_cents", "date", "time_seconds"]).head())

Bank transactions (prepared):
shape: (4, 4)
┌─────────┬──────────────┬────────────┬──────────────┐
│ bank_id ┆ amount_cents ┆ date       ┆ time_seconds │
│ ---     ┆ ---          ┆ ---        ┆ ---          │
│ str     ┆ i64          ┆ date       ┆ i16          │
╞═════════╪══════════════╪════════════╪══════════════╡
│ B001    ┆ 150000       ┆ 2024-10-01 ┆ 32306        │
│ B002    ┆ 230050       ┆ 2024-10-01 ┆ -15086       │
│ B003    ┆ 89025        ┆ 2024-10-02 ┆ -25847       │
│ B004    ┆ 150000       ┆ 2024-10-02 ┆ -7908        │
└─────────┴──────────────┴────────────┴──────────────┘


### Step 2: Match with Tolerance Window

**Fuzzy matching** allows matching records that are similar but not exactly identical.

In [11]:
# Cross join on same date + amount (exact)
matches = (
    bank_prep
    .join(
        ledger_prep,
        on=["date", "amount_cents"],
        how="inner"
    )
    # Filter: timestamps within 60 seconds
    .filter(
        (pl.col("time_seconds") - pl.col("time_seconds_right")).abs() <= 60
    )
    .select([
        pl.col("bank_id"),
        pl.col("ledger_id"),
        pl.col("amount"),
        pl.col("timestamp").alias("bank_time"),
        pl.col("timestamp_right").alias("ledger_time"),
        (pl.col("timestamp") - pl.col("timestamp_right"))
          .dt.total_seconds()
          .alias("time_diff_seconds")
    ])
)

print("Matched transactions:")
print(matches)

Matched transactions:
shape: (2, 6)
┌─────────┬───────────┬────────┬─────────────────────┬─────────────────────┬───────────────────┐
│ bank_id ┆ ledger_id ┆ amount ┆ bank_time           ┆ ledger_time         ┆ time_diff_seconds │
│ ---     ┆ ---       ┆ ---    ┆ ---                 ┆ ---                 ┆ ---               │
│ str     ┆ str       ┆ f64    ┆ datetime[μs]        ┆ datetime[μs]        ┆ i64               │
╞═════════╪═══════════╪════════╪═════════════════════╪═════════════════════╪═══════════════════╡
│ B001    ┆ L001      ┆ 1500.0 ┆ 2024-10-01 09:15:30 ┆ 2024-10-01 09:16:00 ┆ -30               │
│ B002    ┆ L002      ┆ 2300.5 ┆ 2024-10-01 14:22:10 ┆ 2024-10-01 14:22:15 ┆ -5                │
└─────────┴───────────┴────────┴─────────────────────┴─────────────────────┴───────────────────┘


### Step 3: Flag Unmatched (Anti-Join)

**Anti-join** returns rows from the left table that have NO match in the right table.

In [12]:
# Bank transactions without a match
unmatched_bank = (
    bank_prep
    .join(matches, on="bank_id", how="anti")
    .select(["bank_id", "amount", "timestamp", "description"])
    .with_columns(pl.lit("UNMATCHED_BANK").alias("status"))
)

# Ledger entries without a match
unmatched_ledger = (
    ledger_prep
    .join(matches, on="ledger_id", how="anti")
    .select(["ledger_id", "amount", "timestamp", "reference"])
    .with_columns(pl.lit("UNMATCHED_LEDGER").alias("status"))
)

print("Unmatched bank transactions:")
print(unmatched_bank)

print("\nUnmatched ledger entries:")
print(unmatched_ledger)

Unmatched bank transactions:
shape: (2, 5)
┌─────────┬────────┬─────────────────────┬───────────────┬────────────────┐
│ bank_id ┆ amount ┆ timestamp           ┆ description   ┆ status         │
│ ---     ┆ ---    ┆ ---                 ┆ ---           ┆ ---            │
│ str     ┆ f64    ┆ datetime[μs]        ┆ str           ┆ str            │
╞═════════╪════════╪═════════════════════╪═══════════════╪════════════════╡
│ B003    ┆ 890.25 ┆ 2024-10-02 11:05:45 ┆ Refund        ┆ UNMATCHED_BANK │
│ B004    ┆ 1500.0 ┆ 2024-10-02 16:30:20 ┆ Invoice #1234 ┆ UNMATCHED_BANK │
└─────────┴────────┴─────────────────────┴───────────────┴────────────────┘

Unmatched ledger entries:
shape: (1, 5)
┌───────────┬────────┬─────────────────────┬───────────┬──────────────────┐
│ ledger_id ┆ amount ┆ timestamp           ┆ reference ┆ status           │
│ ---       ┆ ---    ┆ ---                 ┆ ---       ┆ ---              │
│ str       ┆ f64    ┆ datetime[μs]        ┆ str       ┆ str              │
╞═══

### Step 4: Validation Rules

In [13]:
def validate_reconciliation(
    bank_df: pl.DataFrame,
    ledger_df: pl.DataFrame,
    matches_df: pl.DataFrame
) -> List[Tuple[str, bool, str]]:
    """Return list of (rule_name, passed, message)"""

    results = []

    # Rule 1: Total matched amounts should equal
    bank_matched_total = matches_df.select(pl.col("amount").sum())[0, 0]
    ledger_matched_total = matches_df.select(pl.col("amount").sum())[0, 0]

    rule1_pass = abs(bank_matched_total - ledger_matched_total) < 0.01
    results.append((
        "Matched totals equal",
        rule1_pass,
        f"Bank: ${bank_matched_total:.2f} | Ledger: ${ledger_matched_total:.2f}"
    ))

    # Rule 2: No duplicate matches (one bank -> one ledger)
    bank_dups = (
        matches_df
        .group_by("bank_id")
        .agg(pl.count().alias("match_count"))
        .filter(pl.col("match_count") > 1)
    )

    rule2_pass = len(bank_dups) == 0
    results.append((
        "No duplicate bank matches",
        rule2_pass,
        f"Found {len(bank_dups)} duplicates" if not rule2_pass else "OK"
    ))

    # Rule 3: Match rate > 80%
    total_bank = len(bank_df)
    matched_bank = len(matches_df)
    match_rate = matched_bank / total_bank if total_bank > 0 else 0

    rule3_pass = match_rate >= 0.80
    results.append((
        "Match rate >= 80%",
        rule3_pass,
        f"{match_rate:.1%} ({matched_bank}/{total_bank})"
    ))

    # Rule 4: All matched timestamps within 5 minutes
    max_diff = matches_df.select(pl.col("time_diff_seconds").abs().max())[0, 0]

    rule4_pass = max_diff <= 300
    results.append((
        "Time differences <= 5min",
        rule4_pass,
        f"Max: {max_diff:.0f}s"
    ))

    return results

# Run validation
validation_results = validate_reconciliation(bank, ledger, matches)

print("\n=== VALIDATION RESULTS ===")
for rule, passed, msg in validation_results:
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {rule}: {msg}")


=== VALIDATION RESULTS ===
[PASS] Matched totals equal: Bank: $3800.50 | Ledger: $3800.50
[PASS] No duplicate bank matches: OK
[FAIL] Match rate >= 80%: 50.0% (2/4)
[PASS] Time differences <= 5min: Max: 30s


/var/folders/cd/l_bbk7dd2fbdr7b_mtcr971m0000gn/T/ipykernel_74379/3620730405.py:25: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("match_count"))


### Step 5: Write Reconciliation Report

In [14]:
# Combine all results
# Ensure all amount columns are Float64 to avoid schema mismatch
reconciliation_report = pl.concat([
    matches.with_columns(pl.lit("MATCHED").alias("status")),
    unmatched_bank.select([
        pl.col("bank_id"),
        pl.lit(None).cast(pl.String).alias("ledger_id"),
        pl.col("amount").cast(pl.Float64),
        pl.col("timestamp").alias("bank_time"),
        pl.lit(None).cast(pl.Datetime).alias("ledger_time"),
        pl.lit(None).cast(pl.Float64).alias("time_diff_seconds"),
        pl.col("status")
    ]),
    unmatched_ledger.select([
        pl.lit(None).cast(pl.String).alias("bank_id"),
        pl.col("ledger_id"),
        pl.col("amount").cast(pl.Float64),
        pl.lit(None).cast(pl.Datetime).alias("bank_time"),
        pl.col("timestamp").alias("ledger_time"),
        pl.lit(None).cast(pl.Float64).alias("time_diff_seconds"),
        pl.col("status")
    ])
], how="vertical_relaxed").sort("bank_time", "ledger_time", nulls_last=True)

print("Reconciliation report:")
print(reconciliation_report)

# Write to Parquet
reconciliation_report.write_parquet(
    "data/curated/reconciliation/2024-10-01.parquet",
    compression="snappy"
)

# Also write validation results
validation_df = pl.DataFrame({
    "rule": [r[0] for r in validation_results],
    "passed": [r[1] for r in validation_results],
    "message": [r[2] for r in validation_results],
    "run_timestamp": [datetime.now()] * len(validation_results)
})

validation_df.write_parquet(
    "data/curated/validation/2024-10-01.parquet",
    compression="snappy"
)

print("\n[OK] Reconciliation report saved to data/curated/reconciliation/")
print("[OK] Validation log saved to data/curated/validation/")

Reconciliation report:
shape: (5, 7)
┌─────────┬───────────┬────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ bank_id ┆ ledger_id ┆ amount ┆ bank_time      ┆ ledger_time    ┆ time_diff_seco ┆ status         │
│ ---     ┆ ---       ┆ ---    ┆ ---            ┆ ---            ┆ nds            ┆ ---            │
│ str     ┆ str       ┆ f64    ┆ datetime[μs]   ┆ datetime[μs]   ┆ ---            ┆ str            │
│         ┆           ┆        ┆                ┆                ┆ f64            ┆                │
╞═════════╪═══════════╪════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ B001    ┆ L001      ┆ 1500.0 ┆ 2024-10-01     ┆ 2024-10-01     ┆ -30.0          ┆ MATCHED        │
│         ┆           ┆        ┆ 09:15:30       ┆ 09:16:00       ┆                ┆                │
│ B002    ┆ L002      ┆ 2300.5 ┆ 2024-10-01     ┆ 2024-10-01     ┆ -5.0           ┆ MATCHED        │
│         ┆           ┆        ┆ 14:22:10       ┆ 14:2

## 6. Streaming Processing for Large Files

**Streaming processing** handles datasets larger than available RAM by processing data in chunks.

In [15]:
# Demonstrate the pattern (with small sample data)
print("Streaming pattern for large datasets:")
print("\n# Bad: OOM risk")
print("all_bank = pl.scan_parquet('bank/*.parquet').collect()")
print("all_ledger = pl.scan_parquet('ledger/*.parquet').collect()")

print("\n# Good: Process in chunks")
print("for month_file in ['2024-01.parquet', '2024-02.parquet', ...]:")
print("    month_matches = (")
print("        pl.scan_parquet(f'bank/{month_file}')")
print("        .join(pl.scan_parquet(f'ledger/{month_file}'), ...)")
print("        .collect()  # Only one month in memory")
print("    )")
print("    month_matches.write_parquet(f'curated/matches/{month_file}')")

print("\nThen query across all months with DuckDB:")
print("duckdb.sql('SELECT * FROM curated/matches/*.parquet').show()")

Streaming pattern for large datasets:

# Bad: OOM risk
all_bank = pl.scan_parquet('bank/*.parquet').collect()
all_ledger = pl.scan_parquet('ledger/*.parquet').collect()

# Good: Process in chunks
for month_file in ['2024-01.parquet', '2024-02.parquet', ...]:
    month_matches = (
        pl.scan_parquet(f'bank/{month_file}')
        .join(pl.scan_parquet(f'ledger/{month_file}'), ...)
        .collect()  # Only one month in memory
    )
    month_matches.write_parquet(f'curated/matches/{month_file}')

Then query across all months with DuckDB:
duckdb.sql('SELECT * FROM curated/matches/*.parquet').show()


## Summary

### When to Use Polars

**(+) Use Polars when:**
- Complex row-level transformations (5+ columns)
- Deduplication with custom logic
- Need to chain 10+ operations
- Working with 10M-500M row datasets
- You want readable code without SQL hell

**(-) Use DuckDB instead when:**
- Simple aggregations (`GROUP BY`, `SUM`)
- Window functions (`LAG`, `LEAD`, `ROW_NUMBER`)
- Querying existing Parquet as-is
- Ad-hoc analysis (faster to write SQL)

**Lesson**: Use them together. Polars for transforms, DuckDB for queries.